In [1]:
###### symbol = 'TSLA'

import yfinance
import numpy as np
import pandas as pd
import pandas_datareader as pdr
import matplotlib.pyplot as plt
# hist = yfinance.download(tickers=symbol,interval="1m", start="2023-01-01", end="2023-01-07")

In [2]:
# from datetime import date, timedelta

# def alldays(year, date_num):
#    d = date(year, 1, 1)                    # January 1st
#    d += timedelta(days = 6 + date_num - d.weekday())  # First Sunday
#    while d.year == year:
#       yield d
#       d += timedelta(days = 7)
# mondays = []
# fridays = []
# for d in alldays(2023,1):
#    mondays.append(d)
# for d in alldays(2023,5):
#     fridays.append(d)



In [3]:
# hist = []
# for i in range(len(mondays)):
#     print(mondays[i],fridays[i])
#     new = yfinance.download(tickers=symbol,interval="1m", start=mondays[i], end=fridays[i])
#     if ~new.empty:
#         hist.append(new)

In [4]:


TODAY = pd.to_datetime("today").date()
START = (TODAY - pd.DateOffset(days=29)).date()

# Reference: https://stackoverflow.com/a/48131963/16051077
d1 = pd.date_range(start=START, end=TODAY, freq="7D")
d2 = d1.shift(6, freq="d")
# fix end date (make sure latest end_date it doesn't go over end_date)
d2 = list(d2)[:-1] + [min(d2[-1], pd.Timestamp(TODAY))]

dates = pd.DataFrame(
    data=dict(start_date=d1, end_date=d2), columns=("start_date", "end_date")
)

df_list = []
for i in dates.index:
    start = dates.at[i, "start_date"]
    end = dates.at[i, "end_date"]

    tickers = ["TSLA"]

    df = yfinance.download(tickers, start=start, end=end, interval="1m")
    df_list.append(df)

hist = pd.concat(df_list)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1 Failed download:
- TSLA: No data found for this date range, symbol may be delisted


In [5]:
hist

,Open,High,Low,Close,Adj Close,Volume
2023-03-13 09:30:00-04:00,167.455002,168.550003,167.380005,168.419998,168.419998,9628682.0
2023-03-13 09:31:00-04:00,168.436493,168.850006,167.940002,168.375000,168.375000,670389.0
2023-03-13 09:32:00-04:00,168.381500,168.403702,165.800003,166.949905,166.949905,1344858.0
2023-03-13 09:33:00-04:00,166.949997,168.466507,166.039993,167.884506,167.884506,1007602.0
2023-03-13 09:34:00-04:00,167.820007,167.865601,166.389999,166.955002,166.955002,635969.0
...,...,...,...,...,...,...
2023-04-06 15:55:00-04:00,184.970001,185.190002,184.870102,185.160004,185.160004,355890.0
2023-04-06 15:56:00-04:00,185.159195,185.259995,185.130005,185.229996,185.229996,323255.0
2023-04-06 15:57:00-04:00,185.225006,185.279999,185.200104,185.260895,185.260895,215198.0
2023-04-06 15:58:00-04:00,185.259995,185.309998,185.210007,185.233994,185.233994,318830.0


In [6]:
# change column name to lower case and without space.
# option 1
new_col_list = []
for i in hist.columns:
    new_col_list.append(i.lower().replace(" ", ""))
hist.columns = new_col_list

In [7]:
# option 2 
# dataframe.columns.values == array
# dataframe.columns == index
# pandas does not want pd.Indexs to be mutable
for i,j in enumerate(hist.columns):
    hist.columns.values[i] = j.lower().replace(" ", "")


In [8]:
# in case zeros value in dataset, change zeros to ones for a better calculation
hist.replace(0,1,inplace=True)

In [9]:


def get_bollinger_bands(data, rate=20, sd=2):
    sma = data['close'].rolling(rate).mean()
    std = data['close'].rolling(rate).std()
    bollinger_up = pd.Series(sma + std * sd, name = 'bollinger_up_' + str(rate)) # Calculate top band
    bollinger_down = pd.Series(sma - std * sd, name ='bollinger_down_' + str(rate)) # Calculate bottom band
    data = data.join([bollinger_up,bollinger_down])
    return data

# symbol = 'AAPL'
# df = pdr.DataReader(symbol, 'yahoo', '2014-07-01', '2015-07-01')
# df.index = np.arange(df.shape[0])




In [10]:
def SMA(data, ndays=20): 
    SMA = pd.Series(data['close'].rolling(ndays).mean(), name = 'SMA_'+ str(ndays)) 
    data = data.join(SMA) 
    return data

In [11]:
def EWMA(data, ndays=20): 
    EMA = pd.Series(data['close'].ewm(span = ndays, min_periods = ndays - 1).mean(), 
                 name = 'EWMA_' + str(ndays)) 
    data = data.join(EMA) 
    return data

In [12]:
def rsi(data, periods = 20):
    
    close_delta = data['close'].diff()

    # Make two series: one for lower closes and one for higher closes
    up = close_delta.clip(lower=0)
    down = -1 * close_delta.clip(upper=0)
    
    ma_up = up.ewm(com = periods - 1, adjust=True, min_periods = periods).mean()
    ma_down = down.ewm(com = periods - 1, adjust=True, min_periods = periods).mean()

    rsi = ma_up / ma_down
    rsi = 100 - (100/(1 + rsi))
    RSI = pd.Series(rsi, name='RSI_' + str(periods))
    data = data.join(RSI)
    return data

In [13]:
def gain(x):
    return ((x > 0) * x).sum()


def loss(x):
    return ((x < 0) * x).sum()


# Calculate money flow index
def mfi(data, n=20):
    high = data['high']
    low = data['low']
    close = data['close']
    volume = data['volume']
    typical_price = (high + low + close)/3
    money_flow = typical_price * volume
    mf_sign = np.where(typical_price > typical_price.shift(1), 1, -1)
    signed_mf = money_flow * mf_sign
    mf_avg_gain = signed_mf.rolling(n).apply(gain, raw=True)
    mf_avg_loss = signed_mf.rolling(n).apply(loss, raw=True)
    mfi = (100 - (100 / (1 + (mf_avg_gain / abs(mf_avg_loss)))))
    MFI = pd.Series(mfi, name='MFI_' + str(n))
    data = data.join(MFI)
    return data

In [14]:
def percent_change(data, level = 'close'):
    PER = pd.Series((data[level] - data[level].shift(1))/data[level].shift(1)*100, name = 'PER_'+level)
    data = data.join(PER)
    return data
    

In [15]:
list(range(7,28,7))

[7, 14, 21]

In [16]:
hist

,open,high,low,close,adjclose,volume
2023-03-13 09:30:00-04:00,167.455002,168.550003,167.380005,168.419998,168.419998,9628682.0
2023-03-13 09:31:00-04:00,168.436493,168.850006,167.940002,168.375000,168.375000,670389.0
2023-03-13 09:32:00-04:00,168.381500,168.403702,165.800003,166.949905,166.949905,1344858.0
2023-03-13 09:33:00-04:00,166.949997,168.466507,166.039993,167.884506,167.884506,1007602.0
2023-03-13 09:34:00-04:00,167.820007,167.865601,166.389999,166.955002,166.955002,635969.0
...,...,...,...,...,...,...
2023-04-06 15:55:00-04:00,184.970001,185.190002,184.870102,185.160004,185.160004,355890.0
2023-04-06 15:56:00-04:00,185.159195,185.259995,185.130005,185.229996,185.229996,323255.0
2023-04-06 15:57:00-04:00,185.225006,185.279999,185.200104,185.260895,185.260895,215198.0
2023-04-06 15:58:00-04:00,185.259995,185.309998,185.210007,185.233994,185.233994,318830.0


In [17]:
length = range(7,28,7)
st = range(1,4,1)

final_table = []
for ii in length:
    for jj in st:
        sub = hist.copy()
    #     hist = mfi(hist,14)

    #     hist = rsi(hist, 14)

        sub = get_bollinger_bands(sub, ii,jj)

    #     hist = percent_change(hist, 'close')

    #     hist = percent_change(hist, 'volume')

        sub['under_BBL'] = np.where(sub['close'] < sub['bollinger_down_'+str(ii)], 1, 0)

        tb = [0,0]
        for i in np.unique(sub.index.date):
            print(i,ii,jj)
            daily_hist = sub[str(i):str(i)]

#             plt.figure(figsize=(18,8))
#             plt.title(symbol + ' Bollinger Bands')
#             plt.xlabel('Days')
#             plt.ylabel('Closing Prices')
#             plt.plot(daily_hist['close'], label='Closing Prices')
#             plt.plot(daily_hist['bollinger_up_'+str(ii)], label='Bollinger Up', c='g')
#             plt.plot(daily_hist['bollinger_down_'+str(ii)], label='Bollinger Down', c='r')




            for i in range(5):
                daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
            daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
            daily_hist['success'] = np.where((daily_hist['under_BBL'] == 1) & (daily_hist['close']*1.001 < daily_hist['highest']), 1,0)

            try:
                print(len(daily_hist[daily_hist['under_BBL']==1]), len(daily_hist[daily_hist['success']==1]), len(daily_hist[daily_hist['success']==1])/len(daily_hist[daily_hist['under_BBL']==1]))
            except ZeroDivisionError:
                print(len(daily_hist[daily_hist['under_BBL']==1]), len(daily_hist[daily_hist['success']==1]), 0)
                
#             plt.scatter(daily_hist[daily_hist['success']==1]['close'].index, daily_hist[daily_hist['success']==1]['close'],label='success', marker = '^', color = 'r')
#             plt.scatter(daily_hist[(daily_hist['success']==0) & (daily_hist['under_BBL']==1)]['close'].index, daily_hist[(daily_hist['success']==0) & (daily_hist['under_BBL']==1)]['close'],label='fail', marker = 'v', color = 'b')
#             plt.legend()
#             plt.show()



            tb[0] +=len(daily_hist[daily_hist['under_BBL']==1])
            tb[1] +=len(daily_hist[daily_hist['success']==1])
        final_table.append([ii,jj,tb[0],tb[1]])


2023-03-13 7 1
81 68 0.8395061728395061
2023-03-14 7 1
84 60 0.7142857142857143
2023-03-15 7 1


<ipython-input-17-0d0a5f45318f>:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
<ipython-input-17-0d0a5f45318f>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
<ipython-input-17-0d0a5f45318f>:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

114 87 0.7631578947368421
2023-03-16 7 1
86 56 0.6511627906976745
2023-03-17 7 1
96 71 0.7395833333333334
2023-03-20 7 1
85 65 0.7647058823529411
2023-03-21 7 1
73 54 0.7397260273972602
2023-03-22 7 1
103 73 0.7087378640776699
2023-03-23 7 1
121 91 0.7520661157024794
2023-03-24 7 1
92 75 0.8152173913043478
2023-03-27 7 1
96 65 0.6770833333333334
2023-03-28 7 1
96 66 0.6875
2023-03-29 7 1
79 53 0.6708860759493671
2023-03-30 7 1
92 46 0.5
2023-03-31 7 1
49 42 0.8571428571428571
2023-04-03 7 1
104 70 0.6730769230769231
2023-04-04 7 1
101 61 0.6039603960396039
2023-04-05 7 1
103 72 0.6990291262135923
2023-04-06 7 1
100 64 0.64
2023-03-13 7 2
4 3 0.75
2023-03-14 7 2
4 4 1.0
2023-03-15 7 2
6 4 0.6666666666666666
2023-03-16 7 2
1 1 1.0
2023-03-17 7 2
4 2 0.5
2023-03-20 7 2
5 5 1.0
2023-03-21 7 2
1 1 1.0
2023-03-22 7 2
6 5 0.8333333333333334
2023-03-23 7 2
3 3 1.0
2023-03-24 7 2
3 2 0.6666666666666666
2023-03-27 7 2
5 4 0.8
2023-03-28 7 2
3 1 0.3333333333333333
2023-03-29 7 2
3 3 1.0
2023-03-3

In [18]:
sub

,open,high,low,close,adjclose,volume,bollinger_up_21,bollinger_down_21,under_BBL
2023-03-13 09:30:00-04:00,167.455002,168.550003,167.380005,168.419998,168.419998,9628682.0,NaN,NaN,0
2023-03-13 09:31:00-04:00,168.436493,168.850006,167.940002,168.375000,168.375000,670389.0,NaN,NaN,0
2023-03-13 09:32:00-04:00,168.381500,168.403702,165.800003,166.949905,166.949905,1344858.0,NaN,NaN,0
2023-03-13 09:33:00-04:00,166.949997,168.466507,166.039993,167.884506,167.884506,1007602.0,NaN,NaN,0
2023-03-13 09:34:00-04:00,167.820007,167.865601,166.389999,166.955002,166.955002,635969.0,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...
2023-04-06 15:55:00-04:00,184.970001,185.190002,184.870102,185.160004,185.160004,355890.0,185.490382,184.772240,0
2023-04-06 15:56:00-04:00,185.159195,185.259995,185.130005,185.229996,185.229996,323255.0,185.501443,184.772597,0
2023-04-06 15:57:00-04:00,185.225006,185.279999,185.200104,185.260895,185.260895,215198.0,185.514308,184.780625,0
2023-04-06 15:58:00-04:00,185.259995,185.309998,185.210007,185.233994,185.233994,318830.0,185.491290,184.793862,0


In [19]:
ttable = pd.DataFrame(final_table, columns = ['length','std','total','success'])

In [20]:
ttable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   length   9 non-null      int64
 1   std      9 non-null      int64
 2   total    9 non-null      int64
 3   success  9 non-null      int64
dtypes: int64(4)
memory usage: 416.0 bytes


In [21]:
ttable['s_rate'] = ttable['success']/ttable['total']

In [22]:
ttable

,length,std,total,success,s_rate
0,7,1,1755,1239,0.705983
1,7,2,74,55,0.743243
2,7,3,0,0,NaN
3,14,1,1839,1279,0.695487
4,14,2,315,225,0.714286
5,14,3,2,2,1.000000
6,21,1,1867,1287,0.689341
7,21,2,406,285,0.701970
8,21,3,16,11,0.687500


In [23]:
ttable['expectation'] = 1.001**ttable['success']*0.999**(ttable['total']-ttable['success'])

In [24]:
ttable

,length,std,total,success,s_rate,expectation
0,7,1,1755,1239,0.705983,2.058799
1,7,2,74,55,0.743243,1.036618
2,7,3,0,0,NaN,1.000000
3,14,1,1839,1279,0.695487,2.050494
4,14,2,315,225,0.714286,1.144357
5,14,3,2,2,1.000000,1.002001
6,21,1,1867,1287,0.689341,2.026007
7,21,2,406,285,0.701970,1.177975
8,21,3,16,11,0.687500,1.006010


In [25]:
# print('Success rate is ' + str(final_table['success'].sum()/final_table['total'].sum()*100))

In [26]:
# final_table['success'].sum(), final_table['total'].sum()

In [27]:
# for i in np.unique(hist.index.date):
#     print(i)
#     sub_hist = hist[str(i):str(i)]

    
#     fig = plt.figure(figsize=(10, 7))

#     # Define position of 1st subplot
#     ax = fig.add_subplot(4, 1, 1)

#     # Set the title and axis labels
#     plt.title(symbol + ' Bollinger Bands')
#     plt.xlabel('Days')
#     plt.ylabel('Closing Prices')
#     plt.plot(sub_hist['close'], label='Closing Prices')
#     plt.plot(sub_hist['bollinger_up_14'], label='Bollinger Up')
#     plt.plot(sub_hist['bollinger_down_14'], label='Bollinger Down')
#     plt.legend()

#     # Define position of 2nd subplot
#     bx = fig.add_subplot(4, 1, 2)

#     # Set the title and axis labels
#     plt.title('Relative Strength Index')
#     plt.xlabel('Date')
#     plt.ylabel('RSI values')
    
#     plt.plot(sub_hist['MFI_14'], label='MFI_14')
#     plt.axhline(y=30, color='grey', linestyle='-')
#     plt.axhline(y=70, color='grey', linestyle='-')
#     plt.plot(sub_hist['RSI_14'], label='RSI_14') 

#     # Add a legend to the axis
#     plt.legend()

    
    
#     cx = fig.add_subplot(4, 1, 3)
#     plt.title('Pecentage Close Change')
#     plt.xlabel('Date')
#     plt.ylabel('Percentage values')
    
#     plt.plot(sub_hist['PER_close'], label='Percent close')

#     plt.legend()
    

#     cx = fig.add_subplot(4, 1, 4)
#     plt.title('Pecentage Volume Change')
#     plt.xlabel('Date')
#     plt.ylabel('Percentage values')
    
#     plt.plot(sub_hist['PER_volume'], label='Percent close')

#     plt.legend()
    
    
#     plt.tight_layout()
#     plt.show()
    
    
   

In [28]:
# for i in np.unique(hist.index.date):
#     print(i)
#     sub_hist = hist[str(i):str(i)]

#     for i in range(5):
#         sub_hist['high_'+str(i)] = sub_hist['high'].shift(-1-i)
#     sub_hist['highest'] = sub_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
#     sub_hist['success'] = np.where(sub_hist['close']*1.001 < sub_hist['highest'], 1,0) # 0.1%
#     print(sub_hist)